# 01 — Target Selection: *E. coli* L-asparaginase II (PDB 3ECA)

This notebook downloads the crystal structure of **E. coli L-asparaginase II** ([PDB 3ECA](https://www.rcsb.org/structure/3ECA)), confirms its homotetrameric assembly, extracts the chain A sequence as our design reference, and saves it as FASTA for notebook 02 (ESM-2 mutation scoring).

**Why this enzyme:** L-asparaginase II is a real oncology drug (marketed as *Elspar*), used to treat acute lymphoblastic leukemia (ALL) — it depletes serum L-asparagine, which leukemia cells depend on externally. Its clinical usefulness is limited by a genuine, documented stability problem: short serum half-life, immunogenicity, and proteolytic degradation (addressed clinically today via PEGylation). That makes it a real test case for AI-guided stability engineering, not a toy example.

**Structural caveat:** the enzyme is a homotetramer (chains A/B/C/D), and the active site is formed *between* subunits — known catalytic residues are Thr12 and Thr89. Any mutation candidate from later notebooks that lands near these residues or the inter-subunit interface needs extra scrutiny, since it could look stabilizing in isolation while disrupting catalysis or tetramer assembly.

## Setup

In [1]:
%pip install -q biopython

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

from Bio import BiopythonWarning
from Bio.PDB import PDBList, PDBParser
from Bio.PDB.Polypeptide import is_aa, protein_letters_3to1

warnings.simplefilter("ignore", BiopythonWarning)

## Download the structure

In [3]:
PDB_ID = "3ECA"
DATA_DIR = Path("../data")
PDB_DIR = DATA_DIR / "pdb"
PDB_DIR.mkdir(parents=True, exist_ok=True)

pdbl = PDBList()
pdb_path = pdbl.retrieve_pdb_file(PDB_ID, pdir=str(PDB_DIR), file_format="pdb")
print("Downloaded to:", pdb_path)

Structure exists: '..\data\pdb\pdb3eca.ent' 
Downloaded to: ..\data\pdb\pdb3eca.ent


## Parse and confirm the tetramer

In [4]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure(PDB_ID, pdb_path)
model = structure[0]

chains = list(model.get_chains())
print(f"Chains found: {[c.id for c in chains]}")

for chain in chains:
    residues = [r for r in chain if is_aa(r, standard=True)]
    print(f"  Chain {chain.id}: {len(residues)} standard amino-acid residues")

assert len(chains) == 4, "Expected a homotetramer (4 chains)"
print("\nConfirmed: homotetramer (A/B/C/D)")
print("Resolution (A):", structure.header.get("resolution"))

Chains found: ['A', 'B', 'C', 'D']
  Chain A: 327 standard amino-acid residues
  Chain B: 327 standard amino-acid residues
  Chain C: 327 standard amino-acid residues
  Chain D: 327 standard amino-acid residues

Confirmed: homotetramer (A/B/C/D)
Resolution (A): 2.4


## Extract the chain A reference sequence

We use chain A's resolved **ATOM** records (not the SEQRES construct sequence) as the reference sequence, so residue numbering matches exactly what's present in the crystal structure. That matters later: notebook 03 folds and structurally validates candidate variants against this same numbering, and the catalytic-residue check below relies on it too.

In [5]:
chain_a = model["A"]
residues_a = [r for r in chain_a if is_aa(r, standard=True)]
sequence = "".join(protein_letters_3to1[r.get_resname()] for r in residues_a)

print(f"Chain A length: {len(sequence)} residues")
print(sequence)

Chain A length: 327 residues
LPNITILATGGTIAGGGDSATKSNYTAGKVGVENLVNAVPQLKDIANVKGEQVVNIGSQDMNDDVWLTLAKKINTDCDKTDGFVITHGTDTMEETAYFLDLTVKCDKPVVMVGAMRPSTSMSADGPFNLYNAVVTAADKASANRGVLVVMNDTVLDGRDVTKTNTTDVATFKSVNYGPLGYIHNGKIDYQRTPARKHTSDTPFDVSKLNELPKVGIVYNYANASDLPAKALVDAGYDGIVSAGVGNGNLYKTVFDTLATAAKNGTAVVRSSRVPTGATTQDAEVDDAKYGFVASGTLNPQKARVLLQLALTQTKDPQQIQQIFNQYD


## Sanity checks

1. Composition: only the 20 standard amino acids appear (no gaps/unknowns marked `X`).
2. The known catalytic residues Thr12 and Thr89 (PDB author numbering) are present where expected — confirming our extraction lines up with the literature.

In [6]:
composition = Counter(sequence)
non_standard = set(composition) - set("ACDEFGHIKLMNPQRSTVWY")
assert not non_standard, f"Unexpected residue letters: {non_standard}"
print("Composition check passed -- only standard amino acids present.")

residue_by_position = {r.id[1]: r.get_resname() for r in residues_a}
for pos in (12, 89):
    print(f"Residue {pos}: {residue_by_position.get(pos)}")
assert residue_by_position.get(12) == "THR", "Expected catalytic Thr12"
assert residue_by_position.get(89) == "THR", "Expected catalytic Thr89"
print("Catalytic residues Thr12 and Thr89 confirmed at expected positions.")

Composition check passed -- only standard amino acids present.
Residue 12: THR
Residue 89: THR
Catalytic residues Thr12 and Thr89 confirmed at expected positions.


## Save as FASTA for notebook 02

In [7]:
DATA_DIR.mkdir(exist_ok=True)
fasta_path = DATA_DIR / "3eca_chainA.fasta"
with open(fasta_path, "w") as f:
    f.write(f">3ECA_A|E.coli_L-asparaginase_II|{len(sequence)}aa\n")
    f.write(sequence + "\n")

print(f"Saved reference sequence to {fasta_path}")

Saved reference sequence to ..\data\3eca_chainA.fasta


## Summary

- Confirmed 3ECA as a homotetramer; chain A gives a 327-residue reference sequence.
- Catalytic Thr12 and Thr89 verified at their expected positions — flagged for extra scrutiny when ranking mutation candidates near the active site / subunit interface in later notebooks.
- Reference sequence saved to `data/3eca_chainA.fasta`.
- **Next:** `02_esm_mutation_scoring.ipynb` — ESM-2 zero-shot masked-marginal scoring of every possible substitution at every position.